## Linear Regression

There is a restaurant franchise and we are considering different cities for opening a new outlet.
- We would like to expand out business to cities that may give your restaurant higher profits.
- The chain already has restaurants in various cities and we have data for profits and populations from the cities.
- We also have data on cities that are candidates for a new restaurant.
- For these cities, we have the city population.


In [ ]:
import copy
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from common import CV_DATA_DIR

def load_data():
    data = np.loadtxt(CV_DATA_DIR / "playground" / "regression" / "ex1data1.txt", delimiter=",")
    X = data[:, 0]
    y = data[:, 1]
    return X, y

### 1 - Load dataset

The dataset for this task:
  - `x_train` is the population of a city
  - `y_train` is the profit of a restaurant in that city. A negative value for profit indicates a loss.

In [ ]:
x_train, y_train = load_data()

### 2 - Examine data

In [ ]:
# Examine X training data
print("Shape of x_train:", x_train.shape)
print("First five elements of x_train are:\n", x_train[:5])
# Examine y training data
print("Shape of y_train:", y_train.shape)
print("First five elements of y_train are:\n", y_train[:5])

In [ ]:
plt.scatter(x_train, y_train, marker='x', c='r')
plt.title("Profits vs. Population per city")
plt.ylabel("Profit in $10,000")
plt.xlabel("Population of City in 10,000s")
plt.show()

### 3 - Compute cost

Gradient descent involves repeated steps to adjust the value of parameters $(w,b)$ to gradually get a smaller and smaller cost $J(w,b)$.
- At each step of gradient descent, it will be helpful to monitor progress by computing the cost $J(w,b)$ as $(w,b)$ gets updated.
- In this section, the function will be implemented to calculate $J(w,b)$ and check the progress of gradient descent implementation.

$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$




In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the cost function for linear regression.
    Args:
        x (ndarray): Shape (m,) Input to the model (Population of cities)
        y (ndarray): Shape (m,) Label (Actual profits for the cities)
        w, b (scalar): Parameters of the model
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    m = len(x)
    return np.sum(((w * x + b) - y) ** 2) / (2 * m)

### 4 - Gradient descent

The gradient descent algorithm is:

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; &
\newline & \rbrace\end{align*}$$

where, parameters $w, b$ are both updated simultaniously and where
$$
\frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{2}
$$
$$
\frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} \tag{3}
$$
* m is the number of training examples in the dataset


*  $f_{w,b}(x^{(i)})$ is the model's prediction, while $y^{(i)}$, is the target value



In [ ]:
def compute_gradient(x, y, w, b):
    """
    Computes the gradient for linear regression
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities)
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b
     """
    m = len(x)
    f_wb = w * x + b
    dj_dw = np.sum((f_wb - y) * x) / m
    dj_db = np.sum(f_wb - y) / m
    return dj_dw, dj_db

In [ ]:
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn theta. Updates theta by taking
    num_iters gradient steps with learning rate alpha

    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in

    for i in range(num_iters):
        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(x, y, w, b)
        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)
        # Print cost every at intervals 10 times or as many iterations if < 10
        if i % math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
    return w, b, J_history, w_history #return w and J,w history for graphing

In [ ]:
initial_w = 0.
initial_b = 0.

iterations = 1500
alpha = 0.01

w, b, _, _ = gradient_descent(
    x_train, y_train,
    initial_w, initial_b,
    compute_cost, compute_gradient, alpha, iterations)

print("w,b found by gradient descent:", w, b)
assert np.isclose(w, +1.16636235)
assert np.isclose(b, -3.63029143)

**Expected Output**:
<table>
  <tr>
    <td> <b> w, b found by gradient descent<b></td>
    <td> 1.16636235 -3.63029143940436</td>
  </tr>
</table>

### 5 - Plot predicted values



In [ ]:
m = x_train.shape[0]
predicted = np.zeros(m)
for i in range(m):
    predicted[i] = w * x_train[i] + b

In [ ]:
plt.plot(x_train, predicted, c = "b")
plt.scatter(x_train, y_train, marker='x', c='r')
plt.title("Profits vs. Population per city")
plt.ylabel('Profit in $10,000')
plt.xlabel('Population of City in 10,000s')